# 가장 기본적인 챗봇 만들기

LLM 을 노드 안에서 호출하는 **가장 단순한 LangGraph 챗봇**을 만든다.

흐름: `START → chatbot → END`. chatbot 노드가 LLM 을 호출해 답한다.
마지막에는 **대화 메모리(checkpointer)** 를 붙여 멀티턴 대화로 확장한다.

## 환경 변수 준비

이 노트북부터는 실제 LLM 을 호출하므로 API 키가 필요하다.
프로젝트 루트의 `.env` 파일에 키를 넣어두고 `load_dotenv()` 로 불러온다.

```
# .env  (.env.example 을 복사해서 작성)
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...   # 웹검색 노트북에서 필요
```

- OpenAI 키: <https://platform.openai.com/api-keys>
- Tavily 키: <https://app.tavily.com>

In [ ]:
import os
from dotenv import load_dotenv

# 프로젝트 루트의 .env 를 찾아 환경변수로 로드
load_dotenv()

# 필요한 키가 있는지 확인
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료:", "OPENAI_API_KEY")

## Step 1. 그래프 State 설정하기

메시지 리스트를 누적하는 State. `add_messages` 리듀서로 대화가 쌓이게 한다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

## Step 2. 챗봇 노드 추가하기

노드는 현재까지의 메시지를 LLM 에 넘기고, 그 응답을 messages 에 추가한다.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)

## Step 3. 엣지 연결 후 컴파일

In [ ]:
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)
graph = graph_builder.compile()

## Step 4. 챗봇 실행하기

먼저 그래프 구조를 시각화한다.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

한 번 호출해보기. (반복 입력 루프 대신 단발 호출로 확인 — 루프 버전은 아래 주석 참고)

In [ ]:
def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

stream_graph_updates("안녕하세요! 한 문장으로 자기소개 해줄래?")

터미널처럼 계속 대화하려면 아래처럼 입력 루프를 돌린다.
(`quit`/`exit`/`q` 입력 시 종료. 이 노트북에는 메모리가 없어 이전 대화를 기억하지 못한다.)

```python
while True:
    user_input = input("User: ")
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    stream_graph_updates(user_input)
```

## Step 5. 메모리 추가하기 (멀티턴)

`MemorySaver` 체크포인터를 붙이면 `thread_id` 단위로 대화 상태가 저장되어 **이전 대화를 기억**한다.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# thread_id 로 대화 세션을 구분한다
config = {"configurable": {"thread_id": "1"}}

같은 `thread_id` 로 두 번 호출 — 두 번째 호출에서 앞 대화를 기억하는지 확인한다.

In [ ]:
res1 = graph.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 스타리야. 기억해줘."}]},
    config=config,
)
print("1턴:", res1["messages"][-1].content)

res2 = graph.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐라고 했지?"}]},
    config=config,
)
print("2턴:", res2["messages"][-1].content)

## 정리

- 가장 단순한 챗봇 = `START → chatbot → END`, 노드 안에서 `llm.invoke`
- `ChatOpenAI(model="gpt-4o")` 로 LLM 준비
- **메모리** = `compile(checkpointer=MemorySaver())` + `config`의 `thread_id` → 멀티턴

다음: 외부 도구(웹검색)를 LLM 이 직접 호출하게 만드는 **Tool Calling**.